# DSA-110 FRB localization

Astrometric analysis of one candidate, run AFTER `scripts/run_localization.py`
has produced, under `<workdir>/<trigname>/`:

* `<trigname>.ms` / `frb_bp.ms` — bandpass-calibrated burst MS
* `<trigname>_full.ms`, `field.ms` — full-file and field MSs
* `<trigname>.B0` — bandpass table
* `rfc_idict.npz` + `icrs_image/*-dirty.fits` — RFC VLBI calibrators (<= 10)
* `images/frb-image.fits`, `images/fimg-image.fits` — wsclean images

Steps: bandpass QC -> field-vs-NVSS astrometry check -> RFC VLBI offset fit
(straight line in RA with intrinsic scatter, ML) -> tclean closeup of the burst
-> corrected FRB position + PS1 cutout.

Run in the `casa38` environment.

In [ ]:
# ---------------- parameters ----------------
trigname = '260715twmx'

workroot = '/media/ubuntu/ssd/localization_processing'
odir = f'{workroot}/{trigname}'

# FRB position guess: None -> peak of images/frb-image.fits (override with a
# SkyCoord if the burst image is confused, e.g.
# SkyCoord('14h14m14.5s', '+15d19m17.0s'))
frbpos_guess = None

# systematic-error ellipse semi-axes (arcsec) drawn on the final plots
ell_a = 1.1 * 2.14
ell_b = 0.7 * 2.14

do_image_full = True   # (re)image the full-file MS for the bright-source check
do_tclean = True       # (re)run the tclean closeup of the burst
do_ps1 = True          # fetch the PS1 colour cutout (needs network + cv2)

In [ ]:
import os
import numpy as np, matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.modeling import models, fitting
from scipy.interpolate import RectBivariateSpline as rbs
from scipy.optimize import minimize

# NVSS catalog
from dsacalib.preprocess import read_nvss_catalog
import pandas as pd
pd.options.mode.chained_assignment = None
df = read_nvss_catalog()
coords = SkyCoord(df.loc[:, 'ra'], df.loc[:, 'dec'], frame='icrs', unit='deg')
maj = df.loc[:, 'major_axis']
flux = df.loc[:, 'flux_20_cm']

## Bandpass-solution quality

In [ ]:
from dsacalib.ms_io import read_caltable
bpvals, bptimes, bpflags, bpa1, bpa2 = read_caltable(
    f'{odir}/{trigname}.B0', cparam=True)
freqs = 1530. - 1024 * 250. / 8192. - np.arange(bpvals.shape[3]) * 250. / 1024.

plt.figure(figsize=(14, 14))
plt.subplots_adjust(wspace=0.3, hspace=0.2)
panels = [('Core', np.setdiff1d(np.arange(102), [20, 21, 22]), 0),
          ('Core', np.setdiff1d(np.arange(102), [20, 21, 22]), 1),
          ('Outriggers', np.arange(15) + 101, 0),
          ('Outriggers', np.arange(15) + 101, 1)]
for ip, (label, ants, pol) in enumerate(panels):
    plt.subplot(2, 2, ip + 1)
    plt.xticks(fontsize=14); plt.yticks(fontsize=14)
    plt.xlim(freqs.min(), freqs.max())
    noants = []
    for ant in ants:
        plt.plot(freqs, np.abs(bpvals[ant, 0, 0, :, pol]), 'k-', alpha=0.2)
        if np.mean(np.abs(bpvals[ant, 0, 0, :, pol])) == 1.:
            noants.append(ant + 1)
    polname = 'B' if pol == 0 else 'A'
    title = f'{label}, pol {polname}'
    if noants:
        title += f' (missing {noants})'
    plt.title(title, fontsize=14)
    if ip % 2 == 0:
        plt.ylabel('Gain amp', fontsize=14)
    if ip >= 2:
        plt.xlabel('Frequency (MHz)', fontsize=14)
plt.show()

## Source-fitting and line-fitting helpers

In [ ]:
def fit_src(image, ira, idec, plot=True, nside=100, method='gaus',
            myvmin=-0.001, myvmax=0.1):
    """Fit the source at pixel (ira, idec); returns (amp, y, x) offsets in
    pixels from that position, or None off-image / on a negative pixel."""
    nont = int(nside / 2)
    if (idec - nont < 0 or idec + nont >= image.shape[0]
            or ira - nont < 0 or ira + nont >= image.shape[1]):
        return None
    mydata = image[idec - nont:idec + nont, ira - nont:ira + nont]
    if not mydata[nont, nont] > 0.:
        return None

    if method == 'gaus':
        gaus = models.Gaussian2D(amplitude=0.03, x_mean=0., y_mean=0.,
                                 x_stddev=3.0, y_stddev=3.0, theta=np.pi / 2.)
        y, x = np.mgrid[:nside, :nside].astype(np.float64)
        y -= nside / 2. + 0.5
        x -= nside / 2. + 0.5
        p = fitting.LevMarLSQFitter()(gaus, x, y, mydata)
        if plot:
            plt.figure(figsize=(10, 5))
            plt.subplot(1, 2, 1)
            plt.imshow(mydata, vmin=myvmin, vmax=myvmax, origin='lower')
            plt.title('Data')
            plt.subplot(1, 2, 2)
            plt.imshow(mydata - p(x, y), vmin=myvmin, vmax=myvmax,
                       origin='lower')
            plt.title('Residual')
            plt.show()
        return p.amplitude.value, p.y_mean.value, p.x_mean.value

    if method == 'interp':
        x = np.arange(nside) - (nside / 2. + 0.5)
        y = np.arange(nside) - (nside / 2. + 0.5)
        p = rbs(x, y, mydata)
        manyx = np.linspace(x.min(), x.max(), nside * 20)
        manyy = np.linspace(y.min(), y.max(), nside * 20)
        interps = p(manyx, manyy)
        wrs = np.where(interps == interps.max())
        if plot:
            plt.figure(figsize=(5, 5))
            plt.imshow(mydata, origin='lower', vmin=myvmin, vmax=myvmax,
                       extent=[x.min(), x.max(), y.min(), y.max()])
            plt.plot([manyy[wrs[1]]], [manyx[wrs[0]]], 'kx')
            plt.title('Data')
            plt.show()
        return interps.max(), manyx[wrs[0]], manyy[wrs[1]]

In [ ]:
def line_model(pars, x):
    return pars[0] * np.array(x) + pars[1]


class StraightLineIntrinsicScatterModel:
    """Straight line + intrinsic scatter, ML fit (flat priors)."""

    def __init__(self, x, y, y_err):
        self.x = np.asarray(x)
        self.y = np.asarray(y)
        self.y_err = np.asarray(y_err)

    def ln_prior(self, pars):
        a, b, lnV = pars
        if a < -2 or a > 2 or b < -10. or b > 10. or lnV < -2. or lnV > 2.:
            return -np.inf
        return 0.0

    def ln_likelihood(self, pars):
        a, b, lnV = pars
        dy = self.y - line_model([a, b], self.x)
        ivar = 1 / (self.y_err ** 2 + np.exp(lnV))
        return -0.5 * (len(self.y) * np.log(2 * np.pi)
                       - np.sum(np.log(ivar)) + np.sum(dy ** 2 * ivar))

    def ln_posterior(self, pars):
        lnp = self.ln_prior(pars)
        if np.isinf(lnp):
            return lnp
        lnprob = lnp + self.ln_likelihood(pars)
        return -np.inf if np.isnan(lnprob) else lnprob

    def __call__(self, pars):
        return self.ln_posterior(pars)

## Field image vs NVSS

In [ ]:
d = fits.open(f'{odir}/images/fimg-image.fits')
image = d[0].data[0, 0, :, :]
w = WCS(d[0].header)
skypos = w.wcs_pix2world(image.shape[0] / 2, image.shape[0] / 2, 0, 0, 0)
pos = SkyCoord([skypos[0]], [skypos[1]], unit=(u.deg, u.deg))

# NVSS selection
flux_lt, maj_lt, rad_lt = 50.0, 20.0, 2.5
idx, d2d, d3d = coords.match_to_catalog_sky(pos)
wrs = np.where((flux > flux_lt) & (maj < maj_lt) & (d2d.deg < rad_lt))[0]
ccoords, fflux, mmaj = coords[wrs], flux[wrs], maj[wrs]

world_coords = np.zeros((len(ccoords), 4))
world_coords[:, 0] = ccoords.ra.value
world_coords[:, 1] = ccoords.dec.value
world_coords[:, 2] = 1.405e9
world_coords[:, 3] = 1.
pix_coords = w.wcs_world2pix(world_coords, 0)
rapix = np.floor(pix_coords[:, 0]).astype(int)
depix = np.floor(pix_coords[:, 1]).astype(int)

fit_amp = np.zeros(len(rapix))
fit_ra = np.zeros(len(rapix))
fit_dec = np.zeros(len(rapix))
for i in range(len(rapix)):
    print('Flux (mJy):', fflux.iloc[i])
    pp = fit_src(image, rapix[i], depix[i], nside=40, method='interp')
    if pp is not None:
        famp, fx, fy = pp
        fit_amp[i], fit_ra[i], fit_dec[i] = famp, -fy, fx

In [ ]:
# diagnostic plot
plt.figure(figsize=(11, 5.))
plt.subplots_adjust(wspace=0.3)
fluxlt = 0.03
scl = 2.   # arcsec / pixel of the fimg image

plt.subplot(121)
plt.xticks(fontsize=14); plt.yticks(fontsize=14)
plt.xlabel('RA offset (arcsec)', fontsize=14)
plt.ylabel('DEC offset (arcsec)', fontsize=14)
fx = scl * fit_ra[fit_amp > fluxlt]
fy = scl * fit_dec[fit_amp > fluxlt]
fflux2 = fflux[fit_amp > fluxlt]
pixlt = 5. * scl
mn1 = np.mean(fx[np.abs(fx) < pixlt]); mn2 = np.mean(fy[np.abs(fy) < pixlt])
plt.scatter(fx, fy, s=fflux2 / 2., color='black')
plt.xlim(pixlt, -pixlt); plt.ylim(-pixlt, pixlt)
print('MEAN position (RA, DEC):', mn1, mn2)
print('UNCERTAINTIES (RA, DEC):', np.std(fx[np.abs(fx) < pixlt]),
      np.std(fy[np.abs(fy) < pixlt]))
plt.grid()

plt.subplot(122)
plt.xticks(fontsize=14); plt.yticks(fontsize=14)
plt.xlabel('NVSS flux (mJy)', fontsize=14)
plt.ylabel('Measured flux (mJy)', fontsize=14)
plt.loglog(fflux, fit_amp * 1e3, 'ko')
plt.plot(fflux, fflux, 'r-')
plt.plot(fflux, np.zeros(len(fflux)) + fluxlt * 1000., 'k-')
plt.grid()
plt.show()

nvss_raoff = fx
nvss_decoff = fy
fflux_final = fflux2

## Bright sources from the full-file MS

In [ ]:
from casatasks import split, applycal, tclean, exportfits

if do_image_full:
    try:
        split(vis=f'{odir}/{trigname}_full.ms', outputvis=f'{odir}/frb_full.ms',
              datacolumn='DATA')
        applycal(vis=f'{odir}/frb_full.ms',
                 gaintable=[f'{odir}/{trigname}.B0'], interp=['nearest'])
    except Exception as exc:
        print(f'cannot (re)make frb_full.ms: {exc}')
    os.system(f'cd {odir}/images; wsclean -weight briggs 0.5 -name full '
              f'-size 6000 6000 -scale 2asec -minuv-l 200 -niter 100 '
              f'-auto-threshold 4.0 -theoretic-beam ../frb_full.ms; cd -')

In [ ]:
d2 = fits.open(f'{odir}/images/full-image.fits')
image2 = d2[0].data[0, 0, :, :]
w2 = WCS(d2[0].header)
skypos2 = w2.wcs_pix2world(image2.shape[0] / 2, image2.shape[0] / 2, 0, 0, 0)
pos2 = SkyCoord([skypos2[0]], [skypos2[1]], unit=(u.deg, u.deg))

flux_lt, maj_lt, rad_lt = 150., 30.0, 2.5
idx, d2d, d3d = coords.match_to_catalog_sky(pos2)
wrs = np.where((flux > flux_lt) & (maj < maj_lt) & (d2d.deg < rad_lt))[0]
ccoords, fflux2b, mmaj = coords[wrs], flux[wrs], maj[wrs]

world_coords = np.zeros((len(ccoords), 4))
world_coords[:, 0] = ccoords.ra.value
world_coords[:, 1] = ccoords.dec.value
world_coords[:, 2] = 1.405e9
world_coords[:, 3] = 1.
pix_coords = w2.wcs_world2pix(world_coords, 0)
rapix = np.floor(pix_coords[:, 0]).astype(int)
depix = np.floor(pix_coords[:, 1]).astype(int)

fit_amp2 = np.zeros(len(rapix))
fit_ra2 = np.zeros(len(rapix))
fit_dec2 = np.zeros(len(rapix))
for i in range(len(rapix)):
    print('Flux (mJy):', fflux2b.iloc[i])
    pp = fit_src(image2, rapix[i], depix[i], nside=40, method='interp',
                 myvmin=-0.00003, myvmax=0.003)
    if pp is not None:
        famp, fx, fy = pp
        fit_amp2[i], fit_ra2[i], fit_dec2[i] = famp, -fy, fx
    else:
        fit_amp2[i] = -1.

In [ ]:
# diagnostic plot: field (black) + full-MS bright sources (red)
plt.figure(figsize=(11, 5))
plt.subplots_adjust(wspace=0.3)
scl = 2.
pixlt = 3. * scl

plt.subplot(121)
plt.xticks(fontsize=14); plt.yticks(fontsize=14)
plt.xlabel('RA offset (arcsec)', fontsize=14)
plt.ylabel('DEC offset (arcsec)', fontsize=14)
plt.scatter(nvss_raoff, nvss_decoff, s=fflux_final / 2., color='black')
plt.scatter(scl * fit_ra2[fit_amp2 > 0.], scl * fit_dec2[fit_amp2 > 0.],
            s=fflux2b[fit_amp2 > 0.] / 3., color='red')
plt.xlim(pixlt, -pixlt); plt.ylim(-pixlt, pixlt)
plt.grid()

plt.subplot(122)
plt.xticks(fontsize=14); plt.yticks(fontsize=14)
plt.xlabel('NVSS flux (mJy)', fontsize=14)
plt.ylabel('Measured flux (mJy)', fontsize=14)
plt.loglog(fflux2b, np.where(fit_amp2 > 0, fit_amp2, np.nan) * 1e3, 'ro')
plt.plot(fflux2b, fflux2b, 'r-')
plt.grid()
plt.show()

## FRB position guess

From the wsclean burst image unless `frbpos_guess` is set above.

In [ ]:
if frbpos_guess is None:
    dfrb = fits.open(f'{odir}/images/frb-image.fits')
    imfrb = dfrb[0].data[0, 0, :, :]
    wfrb = WCS(dfrb[0].header)
    iy, ix = np.unravel_index(np.nanargmax(imfrb), imfrb.shape)
    sky = wfrb.wcs_pix2world([[ix, iy, 1.405e9, 1.]], 0)[0]
    frbpos = SkyCoord(sky[0] * u.deg, sky[1] * u.deg)
    print(f'burst image peak {np.nanmax(imfrb):.4f} Jy at pixel '
          f'({ix}, {iy})')
else:
    frbpos = frbpos_guess
print('FRB position guess:', frbpos.to_string('hmsdms'))

## RFC VLBI calibrator offsets

Fit RA/DEC offsets of the (<= 10) RFC sources vs RA with the
intrinsic-scatter straight-line model, and evaluate at the FRB RA.

In [ ]:
idict = np.load(f'{odir}/rfc_idict.npz', allow_pickle=True)['idict'].item()
filenames = [f'{odir}/icrs_image/{n}-dirty.fits' for n in idict['jname']]
posses = idict['position']
fluxes = idict['flux']
mypixlt = 10.

famp = np.zeros(len(posses))
fra = np.zeros(len(posses))
fdec = np.zeros(len(posses))
myx = np.zeros(len(posses))
myflux = np.zeros(len(posses))

for i, f in enumerate(filenames):
    try:
        print(f)
        d = fits.open(f)
        image = d[0].data[0, 0, :, :]
        w = WCS(d[0].header)
        world_coords = np.array([[posses[i].ra.value, posses[i].dec.value,
                                  1.405e9, 1.]])
        pix_coords = w.wcs_world2pix(world_coords, 0)
        rapix = int(np.floor(pix_coords[0, 0]))
        depix = int(np.floor(pix_coords[0, 1]))
        pp = fit_src(image, rapix, depix, nside=300, method='interp')
        if pp is not None:
            flamp, fx, fy = pp
            famp[i], fra[i], fdec[i] = flamp, -fy[0], fx[0]
            myx[i], myflux[i] = posses[i].ra.deg, fluxes[i]
            if np.hypot(fx[0], fy[0]) > mypixlt:
                famp[i] = fra[i] = fdec[i] = myx[i] = myflux[i] = np.nan
                print('FLAGGING THE ABOVE')
    except Exception:
        famp[i] = fra[i] = fdec[i] = myx[i] = myflux[i] = np.nan

In [ ]:
# clean up data and fit
scl = 0.3   # arcsec / pixel of the icrs images
fluxlt = 0.001
fx = scl * fra[famp > fluxlt]
fy = scl * fdec[famp > fluxlt]
mmyx = myx[famp > fluxlt]
mmyflux = myflux[famp > fluxlt]
frbx = frbpos.ra.deg

ylt = 5.
fx[np.abs(fx - np.median(fx)) > ylt] = np.nan
fy[np.abs(fy - np.median(fy)) > ylt] = np.nan
fx[np.isnan(fy)] = np.nan
fy[np.isnan(fx)] = np.nan
mmyx[np.isnan(fy)] = np.nan
mmyflux[np.isnan(fy)] = np.nan
fx = fx[~np.isnan(fx)]
fy = fy[~np.isnan(fy)]
mmyx = mmyx[~np.isnan(mmyx)]
mmyflux = mmyflux[~np.isnan(mmyflux)]
mmyerr = 0.45 * 25. / (1000. * mmyflux)

scatter_model_x = StraightLineIntrinsicScatterModel(mmyx, fx, mmyerr)
result_ml_scatter_x = minimize(
    lambda *a: -scatter_model_x.ln_likelihood(*a), x0=[0.01, 0., 0.],
    method='BFGS')
print(result_ml_scatter_x)

scatter_model_y = StraightLineIntrinsicScatterModel(mmyx, fy, mmyerr)
result_ml_scatter_y = minimize(
    lambda *a: -scatter_model_y.ln_likelihood(*a), x0=[0.01, 0., 0.],
    method='BFGS')
print(result_ml_scatter_y)

In [ ]:
plt.figure(figsize=(10, 6))
plt.subplot(211)
plt.yticks(fontsize=14); plt.xticks(fontsize=14)
plt.ylabel('RA offset (arcsec)', fontsize=16)
plt.scatter(mmyx, fx, s=mmyflux / 0.01)
plt.errorbar(mmyx, fx, fmt='.', yerr=mmyerr)
plt.plot(mmyx, line_model(result_ml_scatter_x.x[:2], mmyx), '-',
         label=f'Sigma {np.sqrt(np.exp(result_ml_scatter_x.x[2])):.2f}')
plt.ylim(np.median(fx) - ylt, np.median(fx) + ylt)
plt.plot([frbx, frbx], [np.median(fx) - ylt, np.median(fx) + ylt],
         'k--', lw=1.)
plt.legend(fontsize=14); plt.grid()

plt.subplot(212)
plt.xlabel('RA (deg)', fontsize=16)
plt.yticks(fontsize=16); plt.xticks(fontsize=16)
plt.ylabel('DEC offset (arcsec)', fontsize=16)
plt.ylim(np.median(fy) - ylt, np.median(fy) + ylt)
plt.scatter(mmyx, fy, s=mmyflux / 0.01)
plt.errorbar(mmyx, fy, fmt='.', yerr=mmyerr)
plt.plot(mmyx, line_model(result_ml_scatter_y.x[:2], mmyx), '-',
         label=f'Sigma {np.sqrt(np.exp(result_ml_scatter_y.x[2])):.2f}')
plt.plot([frbx, frbx], [np.median(fy) - ylt, np.median(fy) + ylt],
         'k--', lw=1.)
plt.legend(fontsize=14); plt.grid()
plt.show()

myoffset_ra = u.arcsec * line_model(result_ml_scatter_x.x[:2], frbx)
myoffset_dec = u.arcsec * line_model(result_ml_scatter_y.x[:2], frbx)
print('RA offset (arcsec):', myoffset_ra.value)
print('DEC offset (arcsec):', myoffset_dec.value)

vlbi_raoff = fx - line_model(result_ml_scatter_x.x[:2], mmyx)
vlbi_decoff = fy - line_model(result_ml_scatter_y.x[:2], mmyx)
nvss_raoff = nvss_raoff - myoffset_ra.value
nvss_decoff = nvss_decoff - myoffset_dec.value

## Image and localize the FRB

In [ ]:
if do_tclean:
    os.system(f'rm -rf {odir}/images/closeup*')
    tclean(vis=f'{odir}/frb_bp.ms', imagename=f'{odir}/images/closeup_zero',
           uvrange='>0.2klambda', imsize=[1000, 1000],
           cell=['0.3arcsec', '0.3arcsec'], weighting='briggs', robust=0.5,
           niter=0, pblimit=0.0, interactive=False,
           phasecenter=f"J2000 {frbpos.to_string('hmsdms')}")
    tclean(vis=f'{odir}/frb_bp.ms', imagename=f'{odir}/images/closeup',
           uvrange='>0.2klambda', imsize=[1000, 1000],
           cell=['0.3arcsec', '0.3arcsec'], weighting='briggs', robust=0.5,
           niter=30, pblimit=0.0, interactive=False,
           phasecenter=f"J2000 {frbpos.to_string('hmsdms')}")
    exportfits(imagename=f'{odir}/images/closeup_zero.image',
               fitsimage=f'{odir}/images/closeup.fits')
    exportfits(imagename=f'{odir}/images/closeup_zero.psf',
               fitsimage=f'{odir}/images/closeup_psf.fits')
    exportfits(imagename=f'{odir}/images/closeup.image',
               fitsimage=f'{odir}/images/closeup_clean.fits')

In [ ]:
# fit the burst position in the closeup and apply the VLBI tie
scl = 0.3
d3 = fits.open(f'{odir}/images/closeup.fits')
image3 = d3[0].data[0, 0, :, :]
w3 = WCS(d3[0].header)

world_coords = np.array([[frbpos.ra.value, frbpos.dec.value, 1.405e9, 1.]])
pix_coords = w3.wcs_world2pix(world_coords, 0)
rapix = int(np.floor(pix_coords[0, 0]))
depix = int(np.floor(pix_coords[0, 1]))

famp, myfdec, myfra = fit_src(image3, rapix, depix, nside=400,
                              method='interp', myvmin=-0.0002, myvmax=0.002)
fra = -scl * myfra
fdec = scl * myfdec
new_ra = frbpos.ra + fra * u.arcsec
new_dec = frbpos.dec + fdec * u.arcsec

new_frbpos = SkyCoord(new_ra - myoffset_ra, new_dec - myoffset_dec)
print('Corrected FRB position:', new_frbpos[0].to_string('hmsdms'))
print('Amp (Jy):', famp)

In [ ]:
# image, psf, and residuals
fig = plt.figure(figsize=(15, 6))
plt.subplots_adjust(wspace=0.3)

dimg = fits.open(f'{odir}/images/closeup.fits')[0].data[0, 0, :, :]
dpsf = fits.open(f'{odir}/images/closeup_psf.fits')[0].data[0, 0, :, :]

from scipy.ndimage import shift
shift_dpsf = shift(dpsf, [myfdec, myfra])
dresid = dimg - shift_dpsf * famp

x = np.arange(500. * 0.3, -500. * 0.3, -0.3)
y = np.arange(-500. * 0.3, 500. * 0.3, 0.3)
xx, yy = np.meshgrid(x, y)
extent = [500. * 0.3, -500. * 0.3, -500. * 0.3, 500. * 0.3]
levs = [-0.4, -0.2, 0.2, 0.4, 0.6, 0.8, 0.95]
levs2 = [-0.1, -0.05, -0.025, 0.025, 0.05, 0.1]
lt = 80.

for ip, (dat, lev, ttl, vmin, vmax) in enumerate([
        (dpsf, levs, 'PSF', -1., 1.),
        (dimg / famp, levs, 'Image', -1., 1.),
        (dresid, levs2, None, -0.1, 0.1)]):
    ax = fig.add_subplot(1, 3, ip + 1)
    ax.tick_params(axis='both', which='major', labelsize=15)
    ax.set_aspect('equal')
    plt.imshow(dat, vmin=vmin, vmax=vmax, extent=extent, cmap='RdBu',
               origin='lower')
    ax.contour(xx, yy, dat if ip < 2 else dresid / famp, levels=lev,
               colors='k')
    plt.xlim(lt, -lt); plt.ylim(-lt, lt)
    plt.title(ttl or f'Residuals (rms {np.std(dresid / famp):.3f})',
              fontsize=16)

fig.text(0.5, 0.13, 'RA offset (arcsec)', ha='center', va='center',
         fontsize=16)
fig.text(0.07, 0.5, 'DEC offset (arcsec)', ha='center', va='center',
         rotation='vertical', fontsize=16)
plt.show()

## PS1 cutout at the corrected position

Requires network access and `cv2`; skipped otherwise.

In [ ]:
cim_big = None
size = 160
if do_ps1:
    try:
        import requests, cv2
        from astropy.table import Table

        def getimages(ra, dec, filters='grizy'):
            service = 'https://ps1images.stsci.edu/cgi-bin/ps1filenames.py'
            return Table.read(f'{service}?ra={ra}&dec={dec}&filters={filters}',
                              format='ascii')

        def geturl(ra, dec, size=240, filters='grizy'):
            table = getimages(ra, dec, filters=filters)
            url = (f'https://ps1images.stsci.edu/cgi-bin/fitscut.cgi?'
                   f'ra={ra}&dec={dec}&size={size}&format=jpg')
            flist = ['yzirg'.find(x) for x in table['filter']]
            table = table[np.argsort(flist)]
            if len(table) > 3:
                table = table[[0, len(table) // 2, len(table) - 1]]
            for i, param in enumerate(['red', 'green', 'blue']):
                url += f'&{param}={table["filename"][i]}'
            return url

        def adjust_gamma(image, gamma=1.0):
            inv = 1.0 / gamma
            lut = np.array([((i / 255.0) ** inv) * 255
                            for i in np.arange(256)]).astype('uint8')
            return cv2.LUT(image, lut)

        r = requests.get(geturl(new_frbpos[0].ra.deg, new_frbpos[0].dec.deg,
                                size=size, filters='gri'), timeout=60)
        with open(f'{odir}/ps1_tmp.jpg', 'wb') as fl:
            fl.write(r.content)
        cim = cv2.cvtColor(cv2.imread(f'{odir}/ps1_tmp.jpg'),
                           cv2.COLOR_BGR2RGB)
        cim = adjust_gamma(cim, 0.7)
        resize_factor = 4
        cim_big = cv2.resize(cim, (resize_factor * size, resize_factor * size),
                             interpolation=cv2.INTER_CUBIC)
        cim_big = cv2.blur(src=cim_big, ksize=(5, 5))
    except Exception as exc:
        print(f'PS1 cutout unavailable: {exc}')

t = np.linspace(0, 2 * np.pi, 100)
x_el = ell_a * np.cos(t)
y_el = ell_b * np.sin(t)

if cim_big is not None:
    plt.figure(figsize=(8, 8))
    plt.xlim(-size * 0.25 / 2., size * 0.25 / 2.)
    plt.ylim(-size * 0.25 / 2., size * 0.25 / 2.)
    plt.axis('off')
    plt.imshow(cim_big, origin='upper',
               extent=[-size * 0.25 / 2., size * 0.25 / 2.,
                       -size * 0.25 / 2., size * 0.25 / 2.])
    plt.plot(x_el, y_el, color='yellow')
    plt.plot([-size * 0.25 / 2. + 3., -size * 0.25 / 2. + 13.],
             [-size * 0.25 / 2. + 3., -size * 0.25 / 2. + 3.],
             '-', color='white')
    plt.text(-size * 0.25 / 2. + 7., -size * 0.25 / 2. + 4., r"10''",
             fontsize=14, color='white')
    plt.show()

## Publication plot

In [ ]:
dclean = fits.open(f'{odir}/images/closeup_clean.fits')[0].data[0, 0, :, :]

fig = plt.figure(figsize=(10, 10))
plt.subplots_adjust(wspace=0.27, hspace=0.25)
lt = 120.

ax = fig.add_subplot(2, 2, 1)
ax.tick_params(axis='both', which='major', labelsize=15)
ax.set_aspect('equal')
plt.imshow(dpsf, vmin=-1., vmax=1., extent=extent, cmap='RdBu',
           origin='lower')
ax.contour(xx, yy, dpsf, levels=levs, colors='k')
plt.xlim(lt, -lt); plt.ylim(-lt, lt)
plt.title('PSF', fontsize=16)

ax = fig.add_subplot(2, 2, 2)
ax.tick_params(axis='both', which='major', labelsize=15)
ax.set_aspect('equal')
plt.imshow(dclean / np.max(dclean), vmin=-1., vmax=1., extent=extent,
           cmap='RdBu', origin='lower')
ax.contour(xx, yy, dclean / np.max(dclean), levels=levs, colors='k')
plt.xlim(lt, -lt); plt.ylim(-lt, lt)
plt.title('Clean image', fontsize=16)

ax = fig.add_subplot(2, 2, 3)
ax.tick_params(axis='both', which='major', labelsize=15)
ax.set_aspect('equal')
plt.plot(x_el, y_el, color='black')
plt.scatter(nvss_raoff, nvss_decoff, s=np.asarray(fflux_final) / 2.,
            alpha=0.7, label='NVSS offsets')
plt.plot(vlbi_raoff, vlbi_decoff, '+', markersize=20, mew=3., color='red',
         label='VLBI sources')
plt.legend(fontsize=12, ncol=1, loc='lower left')
plt.xlim(2.9, -2.9); plt.ylim(-2.9, 2.9)
plt.title('Offsets', fontsize=16)

if cim_big is not None:
    ax = fig.add_subplot(2, 2, 4)
    ax.tick_params(axis='both', which='major', labelsize=15)
    ax.set_aspect('equal')
    plt.xlim(7., -7.); plt.ylim(-7., 7.)
    plt.imshow(cim_big, origin='upper',
               extent=[size * 0.25 / 2., -size * 0.25 / 2.,
                       -size * 0.25 / 2., size * 0.25 / 2.])
    plt.plot(x_el, y_el, color='yellow', lw=3.)
    plt.title('Localization', fontsize=16)

fig.text(0.5, 0.04, 'Right Ascension offset (arcsec)', ha='center',
         va='center', fontsize=16)
fig.text(0.03, 0.5, 'Declination offset (arcsec)', ha='center', va='center',
         rotation='vertical', fontsize=16)
plt.savefig(f'{odir}/localizations.pdf', bbox_inches='tight')
plt.show()

print('Final position:', new_frbpos[0].to_string('hmsdms'))
print(f'Figure saved to {odir}/localizations.pdf')

To finish up, calculate the total error by adding the fitted intrinsic
scatters in quadrature to the statistical error on the beam, scaled by the
S/N in the FRB image.